# LanceDB fundamentals

In [1]:
import lancedb 
from constants import KNOWLEDGE_BASE_PATH

vector_db = lancedb.connect(KNOWLEDGE_BASE_PATH)
vector_db

LanceDBConnection(uri='c:\\Users\\tasmi\\MLOPS\\llmops_sadia_awan_mlo25\\09_lancedb\\knowledge_base')

In [2]:
import json

with open("animals_text_embeddings.json") as file:
    data= json.load(file)

data

[{'text': 'A small brown dog running.', 'vector': [0.12, 0.85, 0.33]},
 {'text': 'A cat resting quietly on a sofa.', 'vector': [0.4, 0.91, 0.1]},
 {'text': 'A large gray elephant drinking water.',
  'vector': [0.88, 0.22, 0.55]},
 {'text': 'A fast cheetah sprinting across the savannah.',
  'vector': [0.95, 0.12, 0.72]},
 {'text': 'A colorful parrot perched on a branch.',
  'vector': [0.25, 0.66, 0.81]},
 {'text': 'A frog sitting on a lily pad.', 'vector': [0.14, 0.44, 0.27]}]

In [3]:
vector_db.create_table("animals_text", exist_ok=True, data=data)
vector_db["animals_text"]

LanceTable(name='animals_text', version=2, _conn=LanceDBConnection(uri='c:\\Users\\tasmi\\MLOPS\\llmops_sadia_awan_mlo25\\09_lancedb\\knowledge_base'))

In [4]:
vector_db["animals_text"].to_pandas()

,text,vector
0,A small brown dog running.,"[0.12, 0.85, 0.33]"
1,A cat resting quietly on a sofa.,"[0.4, 0.91, 0.1]"
2,A large gray elephant drinking water.,"[0.88, 0.22, 0.55]"
3,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]"
4,A colorful parrot perched on a branch.,"[0.25, 0.66, 0.81]"
5,A frog sitting on a lily pad.,"[0.14, 0.44, 0.27]"
6,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
7,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"


In [5]:
more_data = [
    {"text": "A panda eating bamboo peacefully.", "vector": [0.51, 0.37, 0.82]},
    {"text": "A lion roaring loudly on a rock.", "vector": [0.93, 0.18, 0.41]},
]

vector_db["animals_text"].add(more_data)

AddResult(version=3)

In [6]:
vector_db["animals_text"].to_pandas()

,text,vector
0,A small brown dog running.,"[0.12, 0.85, 0.33]"
1,A cat resting quietly on a sofa.,"[0.4, 0.91, 0.1]"
2,A large gray elephant drinking water.,"[0.88, 0.22, 0.55]"
3,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]"
4,A colorful parrot perched on a branch.,"[0.25, 0.66, 0.81]"
5,A frog sitting on a lily pad.,"[0.14, 0.44, 0.27]"
6,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
7,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"
8,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
9,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"


## Create empty table

In [7]:
from lancedb.pydantic import LanceModel

class Article(LanceModel):
    title: str
    content: str
    
vector_db.create_table("articles", schema=Article, exist_ok=True)

LanceTable(name='articles', version=1, _conn=LanceDBConnection(uri='c:\\Users\\tasmi\\MLOPS\\llmops_sadia_awan_mlo25\\09_lancedb\\knowledge_base'))

In [8]:
vector_db.list_tables()

ListTablesResponse(tables=['animals_text', 'articles'], page_token=None)

TODO: add data to articles table

In [9]:
articles_data = [
    {
        "title": "Fiskar",
        "content": """Fiskar (Pisces) är en grupp vattenlevande ryggradsdjur med fenor, som indelas i benfiskar, broskfiskar och käklösa fiskar. De flesta arter andas med gälar och är växelvarma. Undantaget är lungfiskar. Eftersom landryggradsdjuren släktskapsmässigt är en typ av kvastfeniga fiskar men inte klassificeras som fiskar är begreppet "fiskar" parafyletiskt. Vetenskapen om fiskar kallas iktyologi.""",
    },
    {
        "title": "Krabba",
        "content": """Krabbor (Brachyura) är en delordning av vattenlevande kräftdjur. Vissa arter fiskas som mat, ofta ansedda som delikatesser. Krabbor har tio ben varav det första paret bär gripklor.""",
    },
]

vector_db["articles"].add(articles_data)

AddResult(version=2)

In [10]:
vector_db["articles"].to_pandas()

,title,content
0,Fiskar,Fiskar (Pisces) är en grupp vattenlevande rygg...
1,Krabba,Krabbor (Brachyura) är en delordning av vatten...


# Drop a table

In [11]:
vector_db.drop_table("articles")

In [12]:
vector_db.list_tables() 

ListTablesResponse(tables=['animals_text'], page_token=None)

# Vector search in LanceDb

Flow here:(common approach in many vector database)
1. use embedding model to create vector embeddings for each document
2. use same embedding model to create vector embedding of the query
3. send in the query_vector to search for closest documents

If we want to do RAG-> send in closest documents to LLM to use as context



In [13]:
vector_db["animals_text"].to_pandas()

,text,vector
0,A small brown dog running.,"[0.12, 0.85, 0.33]"
1,A cat resting quietly on a sofa.,"[0.4, 0.91, 0.1]"
2,A large gray elephant drinking water.,"[0.88, 0.22, 0.55]"
3,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]"
4,A colorful parrot perched on a branch.,"[0.25, 0.66, 0.81]"
5,A frog sitting on a lily pad.,"[0.14, 0.44, 0.27]"
6,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
7,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"
8,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
9,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"


In [14]:
query_vector = [0.5, 0.2, 0.9]
vector_db["animals_text"].search(query_vector).limit(3).to_pandas()

df = vector_db["animals_text"].search(query_vector).limit(3).to_pandas()
df

,text,vector,_distance
0,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]",0.0354
1,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]",0.0354
2,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]",0.2413


## Embeddings API in LanceDB

- makes life simpler
- automatically generate embeddings

embedding models can be found here for cohere

- embedding models

In [22]:
from lancedb.embeddings import get_registry
from dotenv import load_dotenv

load_dotenv()

embedding_model = get_registry().get("cohere").create(name = "embed-multilingual-v3.0")

embedding_model.ndims()

1024

In [ ]:
from lancedb.pydantic import Vector, LanceModel

class JokeModel(LanceModel):
    joke: str = embedding_model.SourceField()
    embedding: Vector(embedding_model.ndims()) = embedding_model.VectorField() # type:igonore

vector_db.create_table("jokes", schema=JokeModel, exist_ok=True)
vector_db["jokes"]



LanceTable(name='jokes', version=3, _conn=LanceDBConnection(uri='c:\\Users\\tasmi\\MLOPS\\llmops_sadia_awan_mlo25\\09_lancedb\\knowledge_base'))

In [33]:
import pandas as pd
with open ("jokes.json", encoding="utf8") as file:
    jokes_data = json.load(file)

df_jokes = pd.DataFrame(jokes_data)
df_jokes.head()

,joke
0,Parallel lines have so much in common—it’s sad...
1,"ETL stands for “Extract, Transform, Leave for ..."
2,What do you call a snake that runs your script...
3,"Gold walks into a bar. The bartender says, “Au..."
4,C# devs don’t argue; they just throw exceptions.


In [34]:

vector_db["jokes"].add(df_jokes)

AddResult(version=4)

In [35]:
df_jokes_embeddings = vector_db["jokes"].to_pandas()
df_jokes_embeddings.head()

,joke,embedding
0,Parallel lines have so much in common—it’s sad...,"[-0.0032196045, 0.0011167526, -0.014953613, 0...."
1,"ETL stands for “Extract, Transform, Leave for ...","[0.012634277, 0.005268097, -0.09136963, 0.0809..."
2,What do you call a snake that runs your script...,"[0.019165039, 0.044677734, -0.021530151, 0.028..."
3,"Gold walks into a bar. The bartender says, “Au...","[0.019866943, 0.05999756, 0.012702942, 0.02343..."
4,C# devs don’t argue; they just throw exceptions.,"[0.013122559, 0.002374649, -0.05706787, 0.0608..."


In [36]:
df_jokes_embeddings["embedding"][10] # note dimensions 1024

array([ 0.01750183,  0.02137756, -0.01231384, ...,  0.05526733,
        0.04537964,  0.01043701], shape=(1024,), dtype=float32)

In [37]:
vector_db["jokes"].search("chemistry related joke").to_pandas()

,joke,embedding,_distance
0,I told a chemistry joke… there was no reaction.,"[-0.03274536, 0.018218994, 0.039276123, 0.0012...",0.811271
1,I told a chemistry joke… there was no reaction.,"[-0.03274536, 0.018218994, 0.039276123, 0.0012...",0.811271
2,I told a chemistry joke… there was no reaction.,"[-0.03274536, 0.018218994, 0.039276123, 0.0012...",0.811271
3,Chemists have all the solutions… mostly in bea...,"[0.0013589859, 0.03086853, -0.0012874603, 0.02...",1.002683
4,Chemists have all the solutions… mostly in bea...,"[0.0013589859, 0.03086853, -0.0012874603, 0.02...",1.002683
5,Chemists have all the solutions… mostly in bea...,"[0.0013589859, 0.03086853, -0.0012874603, 0.02...",1.002683
6,Why did the chemist ground his kids? Because t...,"[-0.008056641, 0.031555176, -0.019454956, 0.01...",1.016093
7,Why did the chemist ground his kids? Because t...,"[-0.008056641, 0.031555176, -0.019454956, 0.01...",1.016093
8,Why did the chemist ground his kids? Because t...,"[-0.008056641, 0.031555176, -0.019454956, 0.01...",1.016093
9,Oxygen and Magnesium got together—OMg!,"[-0.04119873, -0.004600525, 0.007701874, 0.006...",1.023621


In [39]:
vector_db["jokes"].search("ge mig kemiskämt").to_pandas()

,joke,embedding,_distance
0,Oxygen and Magnesium got together—OMg!,"[-0.04119873, -0.004600525, 0.007701874, 0.006...",1.149499
1,Oxygen and Magnesium got together—OMg!,"[-0.04119873, -0.004600525, 0.007701874, 0.006...",1.149499
2,Oxygen and Magnesium got together—OMg!,"[-0.04119873, -0.004600525, 0.007701874, 0.006...",1.149499
3,I told a chemistry joke… there was no reaction.,"[-0.03274536, 0.018218994, 0.039276123, 0.0012...",1.177502
4,I told a chemistry joke… there was no reaction.,"[-0.03274536, 0.018218994, 0.039276123, 0.0012...",1.177502
5,I told a chemistry joke… there was no reaction.,"[-0.03274536, 0.018218994, 0.039276123, 0.0012...",1.177502
6,Chemists have all the solutions… mostly in bea...,"[0.0013589859, 0.03086853, -0.0012874603, 0.02...",1.232028
7,Chemists have all the solutions… mostly in bea...,"[0.0013589859, 0.03086853, -0.0012874603, 0.02...",1.232028
8,Chemists have all the solutions… mostly in bea...,"[0.0013589859, 0.03086853, -0.0012874603, 0.02...",1.232028
9,Never trust an atom—they make up everything.,"[-0.008956909, 0.003440857, -0.020614624, 0.04...",1.268420
